In [1]:
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.utils import Sequence
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.callbacks import ModelCheckpoint

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from umap import UMAP
from tqdm import tqdm
from sklearn.metrics import roc_curve, auc

fnames = glob("clips/train/*npy")
testnames = glob('clips/test/*npy')
len(fnames), len(testnames)

(240759, 26615)

In [2]:
N = 1800  # length of 
SAMPLES = 60000

def getbrokenfile(fnames):
    while True:
        f = random.choice(self.fnames)
        x = np.load(f)
        if(x.shape[0]==1_000_000):
            return x
        else:
            pass
 

In [3]:
import math
import random

class MyGenerator(Sequence):
    def __init__(self, fnames, batch_size=64):
        self.fnames = fnames
        self.batch_size= 64
        
    def __len__(self):
        return math.ceil(len(self.fnames) / self.batch_size)
    
    def __getitem__(self, idx):
        x_train = []
        y_train = []
        for i in range(self.batch_size):
            a = getbrokenfile(self.fnames)
            a = a-a.min()
            a = a/a.max()
            idx = random.choice(list(range(0, a.shape[0]-N) ))
            aa = a[idx:idx+N]
            b = getbrokenfile(self.fnames)
            b = b-b.min()
            b = b/b.max()
 
            idx = random.choice(list(range(0, b.shape[0]-N )))
            bb = b[idx:idx+N]
            
            choice = random.choice([0,1])
            y_train.append(choice)
            idx = random.choice(list(range(0, a.shape[0]-N )))
             
            if(choice==0):
                #source is A
                clip = a[idx:idx+N]
            else:
                clip = b[idx:idx+N]
            assert len(clip)==N
             
            x_train.append(np.hstack([aa,bb,clip]))
        x_train = np.array(x_train)
        y_train = np.array(y_train)
        return x_train, y_train     
        
mygen = MyGenerator(random.sample(fnames,SAMPLES))
testgen = MyGenerator(random.sample(testnames, SAMPLES//10))

In [4]:
checkpoint =ModelCheckpoint("best", save_best=True)
def buildABModel(N):
    # size of model is N
    model = Sequential()
    model.add(Dense(768, input_shape=(N*3,), activation='relu'))
    model.add(Dense(300, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(100, activation='relu'))
    model.add(Dense(50, activation='relu'))
    model.add(Dropout(.3))
    model.add(Dense(1, activation='sigmoid'))
    return model
   
model = buildABModel(N)
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['acc'])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 768)               4147968   
                                                                 
 dense_1 (Dense)             (None, 300)               230700    
                                                                 
 batch_normalization (BatchN  (None, 300)              1200      
 ormalization)                                                   
                                                                 
 dense_2 (Dense)             (None, 100)               30100     
                                                                 
 dense_3 (Dense)             (None, 50)                5050      
                                                                 
 dropout (Dropout)           (None, 50)                0         
                                                        

In [ ]:
history = model.fit(mygen,validation_data=testgen, 
                   validation_steps=10,  
                    batch_size=256, epochs=50, callbacks=[es, checkpoint] )
with open("history.pkl", "wb") as fh:
    pkl.dump(history, fh)

Epoch 1/40
 53/938 [>.............................] - ETA: 2:37:47 - loss: 0.5208 - acc: 0.7748

In [ ]:
retestgen=  MyGenerator(random.sample(testnames,15000))

In [ ]:
x_test = []
y_test = []
y_hats = []
for i in range(5):
    x_tmp, y_tmp = retestgen.__getitem__(i)
    y_test.append(y_tmp)
    y_hat= model.predict(x_tmp)
    y_hats.append(y_hat)
y_test = np.hstack(y_test)
y_hats = np.vstack(y_hats)
fp, tp, thresh = roc_curve(y_test, y_hats)
plt.plot(fp,tp)
plt.plot([0,1],[0,1],linestyle='--',c='r')
plt.title(f"AUC: {auc(fp,tp):0.3f}")
plt.grid()

# GOLD Testing
The model has never seen any of the files in the gold directory


In [ ]:
goldfiles = glob("gold*/*flac")
goldfiles

In [ ]:
!rm -rf goldclips
!mkdir goldclips


In [1]:
import librosa
CHUNK = 1_000_000
THRESH = 0.02
peaks=[]
filecount = 0
for fid, f in tqdm(enumerate(goldfiles)):
    x,sr = librosa.load(f, sr = None)
    for i in range(0, len(x), CHUNK):
        y = x[i:i+CHUNK]
        z=librosa.feature.rms(y=x[i:i+CHUNK])[0]
        for j, zz in enumerate(z):
            if(zz>THRESH):
                peaks.append(i+j)
    for p in peaks:
        clip = x[p-CHUNK//2:p+CHUNK//2]
         
        if(random.random()<0.9):
            np.save(f"goldclips/clip_{fid}_{p}.npy", clip)
        else:
            np.save(f"goldclips/clip_{fid}_{p}.npy", clip)
    del x

NameError: name 'tqdm' is not defined